# Sonnet Next-Word Prediction with Train/Validation Split

A follow-up sonnet experiment: the data is split into training and validation sets with scikit-learn, and training uses `EarlyStopping` and `ReduceLROnPlateau` to fight overfitting. The trained model predicts the next word and continues a seed phrase.

**How to run:** Open in Google Colab, choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.

_Rewritten from my original Colab notebook: `Sonnet-last-try.ipynb`._

## 1. Imports

In [ ]:
import re
import urllib.request

import numpy as np
import tensorflow as tf
import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow
print("TensorFlow:", tf.__version__)

from sklearn.model_selection import train_test_split

## 2. Load the sonnets from Project Gutenberg

In [ ]:
def download_gutenberg_book(url):
    """Download a plain-text book from Project Gutenberg and remove the licence header/footer."""
    raw = urllib.request.urlopen(url).read().decode("utf-8-sig")
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    return raw.replace("\r\n", "\n").strip()

In [ ]:
book = download_gutenberg_book("https://www.gutenberg.org/cache/epub/1041/pg1041.txt")

# Every line that has some words but is not just a Roman numeral (sonnet number) is a verse line.
lines = [l.strip() for l in book.split("\n")]
lines = [l for l in lines if l and not re.fullmatch(r"[IVXLC]+", l)]
print("Number of verse lines:", len(lines))
print("\n".join(lines[:14]))

## 3. Vocabulary and n-gram samples

In [ ]:
def clean_words(line):
    """Lower-case a line and keep only words (letters and apostrophes)."""
    line = line.lower().replace("\u2019", "'")
    return re.findall(r"[a-z']+", line)


def build_vocab(lines):
    """Map every word to an integer id. Id 0 is reserved for padding."""
    words = sorted({w for line in lines for w in clean_words(line)})
    word_to_id = {w: i + 1 for i, w in enumerate(words)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word


def encode(line, word_to_id):
    return [word_to_id[w] for w in clean_words(line) if w in word_to_id]

In [ ]:
def make_ngram_dataset(lines, word_to_id, max_len=None):
    """For each line 'a b c d' create the samples [a]->b, [a b]->c, [a b c]->d."""
    sequences = []
    for line in lines:
        ids = encode(line, word_to_id)
        for i in range(1, len(ids)):
            sequences.append(ids[: i + 1])
    if max_len is None:
        max_len = max(len(s) for s in sequences)
    padded = keras.utils.pad_sequences(sequences, maxlen=max_len, padding="pre", truncating="pre")
    X, y = padded[:, :-1], padded[:, -1]
    return X, y, max_len

In [ ]:
word_to_id, id_to_word = build_vocab(lines)
vocab_size = len(word_to_id) + 1
X, y, max_len = make_ngram_dataset(lines, word_to_id)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Vocabulary:", vocab_size)

## 4. Model

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(max_len - 1,)),
    layers.Embedding(vocab_size, 200),
    layers.Bidirectional(layers.LSTM(256, return_sequences=True)),
    layers.Dropout(0.4),
    layers.Bidirectional(layers.LSTM(256)),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 5. Train with callbacks
`ReduceLROnPlateau` lowers the learning rate when validation loss stops improving; `EarlyStopping` stops training and restores the best weights.

In [ ]:
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-5),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=50, batch_size=64, callbacks=callbacks)

## 6. Evaluate

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="train loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss: {loss:.3f} | accuracy: {acc:.3f}")

## 7. Predict

In [ ]:
def predict_next_word(model, seed_text, word_to_id, id_to_word, max_len):
    ids = encode(seed_text, word_to_id)
    padded = keras.utils.pad_sequences([ids], maxlen=max_len - 1, padding="pre", truncating="pre")
    probs = model.predict(padded, verbose=0)[0]
    probs[0] = 0  # never predict the padding id
    return id_to_word.get(int(np.argmax(probs)), "")


def generate_text(model, seed_text, n_words, word_to_id, id_to_word, max_len):
    text = seed_text
    for _ in range(n_words):
        next_word = predict_next_word(model, text, word_to_id, id_to_word, max_len)
        if not next_word:
            break
        text += " " + next_word
    return text

In [ ]:
seed = "From fairest creatures we desire"
print("Next word:", predict_next_word(model, seed, word_to_id, id_to_word, max_len))
print("Continuation:", generate_text(model, seed, 10, word_to_id, id_to_word, max_len))

In [ ]:
model.save("sonnet_lstm_callbacks.keras")